In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

windowed_df = pd.read_csv('../output/windown_sliced_data/windowed_df_10d.csv', parse_dates=['date', 'window_start', 'window_end', 'test_date'])
windowed_df

,date,patient_id,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen,window_start,window_end,test_date,is_test_day
0,2019-04-28,16f4b,0.651879,0.0,1.500000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
1,2019-04-29,16f4b,0.704693,0.0,1.500000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
2,2019-04-30,16f4b,0.603180,0.0,1.000000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
3,2019-05-01,16f4b,0.651631,0.0,1.200000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
4,2019-05-02,16f4b,0.668107,0.0,1.000000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
...,...,...,...,...,...,...,...,...,...,...,...
5407,2019-06-21,ec812,0.661363,0.0,1.125000,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5408,2019-06-22,ec812,0.623560,0.0,0.666667,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5409,2019-06-23,ec812,0.708682,0.0,0.875000,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5410,2019-06-24,ec812,0.656330,0.0,0.888889,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False


In [2]:
PREDICTORS = ['entropy_rate', 'early_warning_score',
              'sleep_quality_score', 'agitation_counts', 'uti_happen']

results = []

for (pid, window_start), group in windowed_df.groupby(['patient_id', 'window_start']):
    baseline = group[group['is_test_day'] == False][PREDICTORS].values
    test_day = group[group['is_test_day'] == True][PREDICTORS].values

    # fit on baseline
    clf = IsolationForest(n_estimators=20, max_samples=10, contamination=0.1, random_state=42)
    clf.fit(baseline)

    # score test day (more negative = more anomalous)
    anomaly_score = clf.decision_function(test_day)[0]
    anomaly_label = clf.predict(test_day)[0]  # -1 = anomaly, 1 = normal

    results.append({
        'patient_id': pid,
        'window_start': window_start,
        'test_date': group[group['is_test_day'] == True]['test_date'].values[0],
        'anomaly_score': anomaly_score,
        'anomaly_label': anomaly_label  # -1 anomaly, 1 normal
    })

if_results = pd.DataFrame(results)
print(if_results['anomaly_label'].value_counts())
print(if_results.head())

anomaly_label
 1    415
-1     77
Name: count, dtype: int64
  patient_id window_start  test_date  anomaly_score  anomaly_label
0      16f4b   2019-04-28 2019-05-08       0.085888              1
1      16f4b   2019-05-03 2019-05-13       0.100597              1
2      1fbe4   2019-04-25 2019-05-05       0.079158              1
3      1fbe4   2019-04-26 2019-05-06       0.056187              1
4      1fbe4   2019-04-27 2019-05-07       0.008614              1


In [3]:
if_results.to_csv("../output/Anomaly_delirium_Revised/FI_anomaly_labels.csv", index=False)
if_results


,patient_id,window_start,test_date,anomaly_score,anomaly_label
0,16f4b,2019-04-28,2019-05-08,0.085888,1
1,16f4b,2019-05-03,2019-05-13,0.100597,1
2,1fbe4,2019-04-25,2019-05-05,0.079158,1
3,1fbe4,2019-04-26,2019-05-06,0.056187,1
4,1fbe4,2019-04-27,2019-05-07,0.008614,1
...,...,...,...,...,...
487,ec812,2019-05-29,2019-06-08,0.067374,1
488,ec812,2019-06-12,2019-06-22,-0.011182,-1
489,ec812,2019-06-13,2019-06-23,0.046199,1
490,ec812,2019-06-14,2019-06-24,0.119338,1


In [4]:
# overall count
n_anomalies = (if_results['anomaly_label'] == -1).sum()
n_normal = (if_results['anomaly_label'] == 1).sum()
print(f"Anomalies: {n_anomalies}")
print(f"Normal: {n_normal}")
print(f"Anomaly rate: {n_anomalies / len(if_results):.2%}")


Anomalies: 77
Normal: 415
Anomaly rate: 15.65%


In [5]:
per_patient = if_results.groupby('patient_id').agg(
    n_windows=('anomaly_label', 'count'),
    n_anomalies=('anomaly_label', lambda x: (x == -1).sum())
).reset_index()

per_patient.to_csv("../output/Anomaly_delirium_Revised/FI_anomaly_rates_per_person.csv", index=False)
per_patient

,patient_id,n_windows,n_anomalies
0,16f4b,2,0
1,1fbe4,39,12
2,30a32,52,13
3,55cd4,50,13
4,93c14,27,3
5,96adf,39,6
6,a2849,47,7
7,c55f8,71,10
8,c5785,56,6
9,c8574,35,1


In [1]:
# calculate anomaly days for each individual

import pandas as pd

df_per_person = pd.read_csv("../output/Anomaly_delirium_Revised/FI_anomaly_rates_per_person.csv")

df_per_person

,patient_id,n_windows,n_anomalies
0,16f4b,2,0
1,1fbe4,39,12
2,30a32,52,13
3,55cd4,50,13
4,93c14,27,3
5,96adf,39,6
6,a2849,47,7
7,c55f8,71,10
8,c5785,56,6
9,c8574,35,1


In [2]:

df_per_person["ratio"] = df_per_person["n_anomalies"] / df_per_person["n_windows"]
df_per_person

,patient_id,n_windows,n_anomalies,ratio
0,16f4b,2,0,0.000000
1,1fbe4,39,12,0.307692
2,30a32,52,13,0.250000
3,55cd4,50,13,0.260000
4,93c14,27,3,0.111111
5,96adf,39,6,0.153846
6,a2849,47,7,0.148936
7,c55f8,71,10,0.140845
8,c5785,56,6,0.107143
9,c8574,35,1,0.028571


In [3]:
df_per_person["ratio"].median()

np.float64(0.1111111111111111)